# E006d 长序列重训（Google AI Pro × Colab / A100）

**预算**：AI Pro 含 Colab Pro 权益 = 100 CU/月（90 天有效），A100 ≈ 15.04 CU/时 → **≈6.6 A100 时/月**。
参考分配：W=512 全程 ≈1 时；W=1024 全程 ≈2 时；余量留给 I094 扩容或调参复跑。

**运行时**：代码单元格 → 更改运行时类型 → **A100**（40GB）。T4/L4 跑得动但不划算，别烧额度。

**断点续跑**：E006d 每 400 步/每 epoch 自动存 `.last.pt`；会话超时/断线后重开本笔记本，
重新执行【训练】格同一条命令即自动续（Ctrl-C 与异常退出也会存盘）。Pro 档无后台执行，
长训请保持前台或靠断点分段；如需挂后台可升 AI Ultra。

**判据**：独立 F1(0.5) > 0.2714（C002b）才接融合；结果回传后按纪律写实验日志+腾讯表格。

In [ ]:
# [1/7] 硬件与环境
!nvidia-smi
!pip install -q "transformers==4.46.3" "peft==0.13.2" scikit-learn pandas tqdm huggingface_hub
import torch
assert torch.cuda.is_available(), "请先切换到 GPU 运行时（建议 A100）"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.0f} GB")

In [ ]:
# [2/7] 装载项目包：把 dpc2026 目录（含 code/ data/ rel/）放到 Drive 的 drop 文件夹后运行
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob
DROP = '/content/drive/MyDrive/dpc2026'   # 你在 Drive 的投放目录
PROJ = '/content/proj'
if os.path.isdir(os.path.join(DROP, 'code')):
    shutil.copytree(DROP, PROJ, dirs_exist_ok=True)
else:
    os.makedirs(PROJ, exist_ok=True)
    for pat in ('code/**/*', 'data/train.csv', 'data/test.csv', 'rel/esm2_tokens_*.npz'):
        for f in glob.glob(os.path.join(DROP, pat), recursive=True):
            dst = os.path.join(PROJ, os.path.relpath(f, DROP))
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(f, dst)
print('项目就位:', PROJ)
!ls {PROJ}/code/E | head

In [ ]:
# [3/7] ESM2-650M 权重落地（约 2.5GB，走 HuggingFace；PLM 权重合规允许）
import os
from huggingface_hub import snapshot_download
dst = '/content/proj/models/esm2_650m'
os.makedirs(dst, exist_ok=True)
snapshot_download('facebook/esm2_t33_650M_UR50D', local_dir=dst,
                  ignore_patterns=['*.pt', '*.bin', '*.h5', '*.ot'])
print(sorted(os.listdir(dst))[:8])

In [ ]:
# [4/7] 分词缓存：W=512 直接用随包的 esm2_tokens_*.npz（宽 512）；W=1024 先重建（约 3-5 分钟）
import os, subprocess, sys
os.chdir('/content/proj')
PY = sys.executable
SCRIPT = 'code/E/E006d-esm2-650-mah-asl-len.py'
# subprocess.run([PY, SCRIPT, '--build-tokens', '--max-len=1024'], check=True)  # 仅 W=1024 时取消注释

In [ ]:
# [5/7] 冒烟（2 步前向+反传，看显存峰值；W=512 预期 <8GB，W=1024 预期 ~20-28GB）
import os, subprocess, sys
os.chdir('/content/proj')
subprocess.run([sys.executable, 'code/E/E006d-esm2-650-mah-asl-len.py',
                '--smoke', '--max-len=512'], check=True)

In [ ]:
# [6/7] 正式训练（断线/超时后重跑本格即续；预算紧先跑 W=512，余量够再跑 W=1024）
import os, subprocess, sys
os.chdir('/content/proj')
cmd = [sys.executable, 'code/E/E006d-esm2-650-mah-asl-len.py',
       '--max-len=512', '--epochs=6']
subprocess.run(cmd, check=False)   # Ctrl-C/断线都会存 .last.pt
# W=1024 版本（先完成 [4/7] 重建分词）：
# cmd = [sys.executable, 'code/E/E006d-esm2-650-mah-asl-len.py',
#        '--max-len=1024', '--epochs=6', '--bs=2', '--accum=8']

In [ ]:
# [7/7] 结果回传 Drive（rel/E006d_report.json + E006d_probs.npz + 断点/最优权重）
import shutil, os, glob, datetime
stamp = datetime.datetime.now().strftime('%m%d_%H%M')
out = f'/content/drive/MyDrive/dpc2026/e006d_out_{stamp}'
os.makedirs(out, exist_ok=True)
for pat in ('rel/E006d_report.json', 'rel/E006d_probs.npz',
            'models/e006d-650-mah-asl-len*.pt'):
    for f in glob.glob('/content/proj/' + pat):
        shutil.copy2(f, out)
print('已回传:', out, os.listdir(out))